## MINGGU-3

### III. Data Wrangling – *Gathering Data*

Tahap ini mengumpulkan data dari sumber utama (API), menggabungkan potongannya menjadi satu dataset, memeriksa struktur awalnya, lalu memvalidasinya terhadap server.

#### 1. Deskripsi Dataset

*MTA Subway Hourly Ridership: Beginning 2025* berisi estimasi jumlah penumpang yang masuk ke stasiun *subway* New York City per jam, menurut metode pembayaran dan kelas tarif. Satu baris = **jam × kompleks stasiun × metode pembayaran × kelas tarif**. Dataset dimiliki MTA, mencakup seluruh New York City, dan diperbarui mingguan.

| Kolom | Tipe | Keterangan |
|---|---|---|
| `transit_timestamp` | *timestamp* | Waktu lokal New York, dibulatkan ke bawah ke jam |
| `transit_mode` | teks | `subway`, `staten_island_railway`, atau `tram` |
| `station_complex_id` | teks | ID kompleks stasiun |
| `station_complex` | teks | Nama kompleks stasiun beserta rutenya |
| `borough` | teks | Wilayah administratif |
| `payment_method` | teks | `omny` atau `metrocard` |
| `fare_class_category` | teks | Kelas tarif |
| `ridership` | numerik | Jumlah penumpang yang masuk |
| `transfers` | numerik | Bagian `ridership` yang masuk lewat transfer gratis |
| `latitude`, `longitude` | numerik | Koordinat stasiun |
| `georeference` | teks | Titik lokasi format WKT `POINT (bujur lintang)` |

#### 2. Sumber Dataset

| Item | Keterangan |
|---|---|
| Nama dan ID | MTA Subway Hourly Ridership: Beginning 2025 (`5wq4-mkjj`) |
| Pemilik | *Metropolitan Transportation Authority* (MTA) |
| Halaman dataset | https://data.ny.gov/d/5wq4-mkjj |
| Akses data | SODA API `https://data.ny.gov/resource/5wq4-mkjj` (`.json` dan `.csv`), tanpa autentikasi |
| Metadata | `https://data.ny.gov/api/views/5wq4-mkjj.json` |
| Lisensi | Kolom lisensi pada metadata kosong |

Dataset *2020-2024* (`wujg-7c2s`) berskema sama dan menyambung tanpa celah, tetapi tidak dimasukkan karena ruang lingkup proyek difiksasi pada data 2025 dan seterusnya. Sumber *streaming* (GTFS-RT) dikumpulkan pada tahap terpisah.

#### 3. Persiapan Lingkungan

Impor pustaka: `requests` untuk API, `pyarrow` untuk CSV dan Parquet, `duckdb` untuk menggabungkan dan menjelajah data besar tanpa memuat semuanya ke RAM, serta `pandas` dan `matplotlib` untuk tabel dan grafik.

In [ ]:
import datetime as dt
import io
import json
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial
from pathlib import Path

import duckdb
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq
import requests
from IPython.display import display
from tqdm import tqdm as _tqdm

print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | pyarrow {pa.__version__} | duckdb {duckdb.__version__}")

Parameter dikumpulkan dalam satu sel. `CUTOFF_TS` dan `EXPECTED` adalah hasil fiksasi *snapshot* (bagian 5) dan tidak diubah, agar hasilnya dapat direproduksi walau server terus menerima data baru. `TEST_MONTHS = None` mengambil seluruh *snapshot* (sekitar 15 sampai 25 menit di Colab); isi mis. `["2025-02"]` untuk uji cepat.

In [ ]:
RESOURCE = "https://data.ny.gov/resource/5wq4-mkjj"         # data (.json / .csv)
VIEW_META = "https://data.ny.gov/api/views/5wq4-mkjj.json"  # metadata

CUTOFF_TS = "2026-09-16T23:00:00"  # snapshot cutoff
EXPECTED = {"rows": 45_243_727, "ridership": 2_238_986_651, "transfers": 111_767_861}

TEST_MONTHS = None  # None = full snapshot
WORKERS = 6         # parallel downloads

BASE = Path("/content/mta_subway")
DAILY_DIR, MONTHLY_DIR, FINAL_DIR = BASE / "01_daily", BASE / "02_monthly", BASE / "03_final"
for folder in (DAILY_DIR, MONTHLY_DIR, FINAL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

Fungsi bantu tampilan: `fmt` memformat bilangan gaya Indonesia, `show_table` menampilkan tabel tanpa indeks, dan `tqdm` diarahkan ke *stdout* agar urutannya sejajar dengan `print`.

In [ ]:
tqdm = partial(_tqdm, file=sys.stdout, ncols=70)
pd.options.display.max_colwidth = 120


def fmt(x, n=0):
    """Format numbers Indonesian-style: 45243727 -> 45.243.727; 10.63 (n=1) -> 10,6."""
    return f"{x:,.{n}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def show_table(df, n=0):
    """Render a DataFrame without index, with dot thousands separators and no line wrapping."""
    return (
        df.style.format(thousands=".", decimal=",", precision=n)
        .hide(axis="index")
        .set_table_styles([{"selector": "th, td", "props": [("white-space", "nowrap")]}])
    )

`soda_get` mengakses API dengan *retry* dan jeda eksponensial untuk galat sementara (kode 429, 5xx, atau *timeout*). Galat 4xx lain berarti kueri salah dan langsung dihentikan.

In [ ]:
def soda_get(url, params=None, retries=8):
    for attempt in range(retries + 1):
        try:
            r = requests.get(url, params=params, timeout=(15, 600))
            if r.status_code == 200:
                return r
            if r.status_code != 429 and r.status_code < 500:
                raise ValueError(f"HTTP {r.status_code}: {r.text[:200]}")  # bad query: do not retry
            reason = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            reason = type(e).__name__
        if attempt == retries:
            raise RuntimeError(f"failed after {retries + 1} attempts: {reason}")
        delay = min(2**attempt, 60)
        print(f"attempt {attempt + 1} failed ({reason}); retrying in {delay}s")
        time.sleep(delay)

`aggregate` menjalankan kueri agregat SoQL di sisi server; dipakai pada bagian 5 dan 9.

In [ ]:
def aggregate(select, where=None, group=None):
    params = {"$select": select, "$limit": 10000}
    if where:
        params["$where"] = where
    if group:
        params["$group"] = params["$order"] = group
    return soda_get(RESOURCE + ".json", params).json()

#### 4. Verifikasi Sumber Data dan Struktur Mentah

##### 4.1 Metadata

Metadata diambil dari endpoint `/api/views` untuk memastikan sumbernya benar.

In [ ]:
meta = soda_get(VIEW_META).json()
info = meta["metadata"]["custom_fields"]["Dataset Summary"]

pd.Series({
    "Name": meta["name"],
    "ID": meta["id"],
    "Owner": meta["attribution"],
    "Period": info["Time Period"],
    "Granularity": info["Granularity"],
    "Update frequency": info["Posting Frequency"],
    "Rows last updated (UTC)": dt.datetime.fromtimestamp(meta["rowsUpdatedAt"], dt.timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "License": meta.get("license") or "(empty)",
}, name="Value").to_frame()

Tipe setiap kolom menurut portal:

In [ ]:
show_table(pd.DataFrame([(c["fieldName"], c["dataTypeName"]) for c in meta["columns"]], columns=["column", "portal type"]))

##### 4.2 Bentuk mentah respons

Pada JSON, seluruh angka dikirim sebagai teks dan `georeference` berupa objek GeoJSON, sehingga kurang cocok untuk pengambilan volume besar.

In [ ]:
sample = soda_get(RESOURCE + ".json", {"$limit": 3, "$order": ":id"}).json()[0]
print(json.dumps(sample, indent=2))
pd.Series({k: type(v).__name__ for k, v in sample.items()}, name="Python type").to_frame()

Endpoint `.csv` lebih ringkas dan memberi `georeference` sebagai teks WKT; endpoint inilah yang dipakai.

In [ ]:
print(soda_get(RESOURCE + ".csv", {"$limit": 3, "$order": ":id"}).text)

##### 4.3 Skema target

Tipe kolom ditetapkan eksplisit agar seluruh potongan seragam. `SORT_KEYS` adalah kunci baris yang dipakai untuk mengurutkan hasil penggabungan.

In [ ]:
SCHEMA = pa.schema([
    ("transit_timestamp", pa.timestamp("us")),
    ("transit_mode", pa.string()),
    ("station_complex_id", pa.string()),
    ("station_complex", pa.string()),
    ("borough", pa.string()),
    ("payment_method", pa.string()),
    ("fare_class_category", pa.string()),
    ("ridership", pa.float64()),
    ("transfers", pa.float64()),
    ("latitude", pa.float64()),
    ("longitude", pa.float64()),
    ("georeference", pa.string()),
])
COLUMNS = SCHEMA.names
SORT_KEYS = ["transit_timestamp", "station_complex_id", "payment_method", "fare_class_category"]

Skema dalam bentuk tabel:

In [ ]:
show_table(pd.DataFrame({"column": COLUMNS, "Arrow type": [str(t) for t in SCHEMA.types]}))

#### 5. Fiksasi *Snapshot* dan Rencana Pengambilan

##### 5.1 Fiksasi *cutoff*

Dataset diperbarui setiap minggu, sehingga dua pengambilan pada waktu berbeda dapat menghasilkan data berbeda. *Snapshot* karena itu didefinisikan sebagai baris dengan `transit_timestamp <= CUTOFF_TS`, yaitu timestamp maksimum saat pengambilan pertama. Baris setelah *cutoff* masuk lewat pembaruan mingguan berikutnya dan tidak termasuk *snapshot* ini. Sumber *streaming* proyek adalah *feed* GTFS-RT. Sel berikut membandingkan status server dengan nilai acuan.

In [ ]:
total = aggregate("count(*) AS n, min(transit_timestamp) AS first_ts, max(transit_timestamp) AS last_ts")[0]
snapshot = aggregate("count(*) AS n", where=f"transit_timestamp <= '{CUTOFF_TS}'")[0]
n_total, n_snapshot = int(total["n"]), int(snapshot["n"])

show_table(pd.DataFrame([
    ("Rows on server now", fmt(n_total)),
    ("Earliest timestamp", total["first_ts"][:19]),
    ("Latest timestamp", total["last_ts"][:19]),
    ("Fixed cutoff", CUTOFF_TS.replace("T", " ")),
    ("Snapshot rows (up to cutoff)", fmt(n_snapshot)),
    ("Rows after cutoff (not fetched)", fmt(n_total - n_snapshot)),
    ("Snapshot equals reference value?", "YES" if n_snapshot == EXPECTED["rows"] else "NO"),
], columns=["Check", "Result"]))

##### 5.2 Rencana pengambilan per hari

Data diambil per hari. Jumlah baris tiap hari ditanyakan ke server lebih dulu, sebagai rencana sekaligus acuan validasi.

In [ ]:
per_day = aggregate("date_trunc_ymd(transit_timestamp) AS ymd, count(*) AS n", where=f"transit_timestamp <= '{CUTOFF_TS}'", group="ymd")
plan = {r["ymd"][:10]: int(r["n"]) for r in per_day if TEST_MONTHS is None or r["ymd"][:7] in TEST_MONTHS}
PLAN_ROWS = sum(plan.values())
print(f"{len(plan)} days, {fmt(PLAN_ROWS)} rows")

Rencana diringkas per bulan, lalu digambarkan:

In [ ]:
plan_df = pd.DataFrame({"day": pd.to_datetime(list(plan)), "rows": list(plan.values())})
plan_df["month"] = plan_df["day"].dt.strftime("%Y-%m")
per_month = plan_df.groupby("month").agg(days=("day", "count"), rows=("rows", "sum")).reset_index()
show_table(per_month)

Grafik jumlah baris per bulan dan per hari:

In [ ]:
fig, (ax_month, ax_day) = plt.subplots(1, 2, figsize=(13, 3.6))
ax_month.bar(per_month["month"], per_month["rows"] / 1e6)
ax_month.set(title="Rows per month", ylabel="million rows")
ax_day.plot(plan_df["day"], plan_df["rows"] / 1e3)
ax_day.set(title="Rows per day", ylabel="thousand rows")
ax_day.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax_day.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
for ax in (ax_month, ax_day):
    ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

#### 6. Proses Pengambilan Data (*Gathering*)

Setiap hari diambil lewat endpoint CSV dengan skema tetap (per halaman 500.000 baris bila perlu), divalidasi terhadap jumlah baris server (diulang hingga tiga kali bila berbeda), lalu disimpan sebagai Parquet. File yang sudah ada dilewati, sehingga proses dapat dilanjutkan bila sesi Colab terputus.

`csv_to_table` mengubah isi CSV menjadi tabel Arrow bertipe sesuai `SCHEMA`.

In [ ]:
def csv_to_table(content):
    options = pacsv.ConvertOptions(column_types=SCHEMA, include_columns=COLUMNS, strings_can_be_null=True)
    return pacsv.read_csv(io.BytesIO(content), convert_options=options).select(COLUMNS)

`fetch_rows` mengambil semua baris yang cocok dengan filter, per halaman, dengan urutan `:id` yang deterministik.

In [ ]:
def day_file(day):
    return DAILY_DIR / day[:7] / f"{day}.parquet"


def fetch_rows(where, page_size=500_000):
    pages, offset = [], 0
    while True:
        params = {"$where": where, "$order": ":id", "$limit": page_size, "$offset": offset}
        table = csv_to_table(soda_get(RESOURCE + ".csv", params).content)
        pages.append(table)
        if table.num_rows < page_size:
            return pa.concat_tables(pages)
        offset += page_size

`download_day` mengunduh satu hari, memvalidasi jumlah barisnya, dan menyimpannya lewat berkas sementara agar file yang tampak selalu utuh. Fungsi mengembalikan lama unduhan (0 bila dilewati).

In [ ]:
def download_day(day, server_rows):
    dest = day_file(day)
    if dest.exists() and pq.ParquetFile(dest).metadata.num_rows == server_rows:
        return 0.0  # already downloaded and complete
    start = dt.date.fromisoformat(day)
    where = (
        f"transit_timestamp >= '{start}T00:00:00' AND transit_timestamp < '{start + dt.timedelta(days=1)}T00:00:00' "
        f"AND transit_timestamp <= '{CUTOFF_TS}'"
    )
    t0 = time.time()
    for _ in range(3):
        table = fetch_rows(where)
        if table.num_rows == server_rows:
            break
        time.sleep(5)
    else:
        raise RuntimeError(f"{day}: got {table.num_rows} rows, server has {server_rows}")
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(".tmp")
    pq.write_table(table, tmp, compression="zstd")
    tmp.replace(dest)
    return time.time() - t0

##### 6.1 Uji pada satu hari

Fungsi diuji pada hari pertama; hasilnya dibaca kembali dari Parquet.

In [ ]:
test_day = min(plan)
download_day(test_day, plan[test_day])

test_df = pq.read_table(day_file(test_day)).to_pandas()
print(f"{test_day}: {fmt(len(test_df))} rows x {test_df.shape[1]} columns (server: {fmt(plan[test_day])})")
test_df.head()

##### 6.2 Pengambilan seluruh hari

Seluruh hari diunduh paralel dengan `WORKERS` *thread*. Bila satu hari gagal, sel berhenti dengan galat dan cukup dijalankan ulang.

In [ ]:
seconds = {}
t0 = time.time()
with ThreadPoolExecutor(WORKERS) as pool:
    futures = {pool.submit(download_day, d, n): d for d, n in plan.items()}
    for f in tqdm(as_completed(futures), total=len(futures), desc="Downloading days", unit="day"):
        seconds[futures[f]] = f.result()
print(f"Finished in {fmt((time.time() - t0) / 60, 1)} minutes")

Ringkasan pengambilan. Jumlah baris dihitung dari metadata file harian; hari yang sudah ada dilewati dan tidak dihitung dalam waktu rata-rata.

In [ ]:
n_daily = sum(pq.ParquetFile(day_file(d)).metadata.num_rows for d in plan)
downloaded = [s for s in seconds.values() if s > 0]

show_table(pd.DataFrame([
    ("Days collected", fmt(len(seconds))),
    ("Downloaded in this run", fmt(len(downloaded))),
    ("Skipped (already present)", fmt(len(seconds) - len(downloaded))),
    ("Rows in daily files", fmt(n_daily)),
    ("Rows on server", fmt(PLAN_ROWS)),
    ("Mean time per downloaded day (s)", fmt(sum(downloaded) / len(downloaded), 1) if downloaded else "-"),
    ("Total size of daily files (MB)", fmt(sum(day_file(d).stat().st_size for d in plan) / 1e6, 1)),
], columns=["Summary", "Value"]))

#### 7. Penggabungan Data

Seluruh potongan berasal dari satu dataset dan berkolom sama, sehingga penggabungannya berupa *concat* vertikal (setara `UNION ALL`); *join* tidak diperlukan. Penggabungan dilakukan berjenjang: contoh dengan `pandas`, lalu harian → bulanan → satu file dengan DuckDB, yang membaca Parquet secara kolumnar dan tidak memuat seluruh data ke RAM.

##### 7.1 Contoh *concat*

Tiga hari pertama digabung dengan `pd.concat`.

In [ ]:
chunks = [pq.read_table(day_file(d)).to_pandas() for d in sorted(plan)[:3]]
combined = pd.concat(chunks, ignore_index=True)

assert len(combined) == sum(map(len, chunks))
print(" + ".join(fmt(len(c)) for c in chunks), "=", fmt(len(combined)), "rows")

##### 7.2 Harian → bulanan

File harian tiap bulan dibaca dengan pola `*.parquet`, diurutkan menurut `SORT_KEYS`, dan ditulis sebagai satu file Parquet. Koneksi DuckDB dibuat lebih dulu (progress bar dimatikan, memori dibatasi).

In [ ]:
con = duckdb.connect()
con.execute("SET enable_progress_bar=false; SET memory_limit='6GB'; SET threads=4")

ORDER_BY = ", ".join(SORT_KEYS)
COPY_OPTS = "FORMAT parquet, COMPRESSION zstd, ROW_GROUP_SIZE 245760"

`merge_month` menggabungkan satu bulan (dilewati bila file bulanan sudah ada dan lengkap) dan mencatat rentang waktunya.

In [ ]:
def month_file(month):
    year, mon = month.split("-")
    return MONTHLY_DIR / f"year={year}" / f"month={mon}" / "part-0.parquet"


def merge_month(month, server_rows):
    dest = month_file(month)
    if not (dest.exists() and pq.ParquetFile(dest).metadata.num_rows == server_rows):
        dest.parent.mkdir(parents=True, exist_ok=True)
        tmp = dest.with_suffix(".tmp")
        con.execute(f"COPY (SELECT * FROM read_parquet('{DAILY_DIR / month}/*.parquet') ORDER BY {ORDER_BY}) TO '{tmp}' ({COPY_OPTS})")
        tmp.replace(dest)
    first_ts, last_ts = con.execute(f"SELECT min(transit_timestamp), max(transit_timestamp) FROM read_parquet('{dest}')").fetchone()
    return {
        "month": month,
        "days": len(list((DAILY_DIR / month).glob("*.parquet"))),
        "rows": pq.ParquetFile(dest).metadata.num_rows,
        "server_rows": server_rows,
        "MB": round(dest.stat().st_size / 1e6, 1),
        "first_ts": first_ts,
        "last_ts": last_ts,
    }

Ringkasan penggabungan per bulan; `rows` harus sama dengan `server_rows`.

In [ ]:
manifest = pd.DataFrame([merge_month(m, n) for m, n in tqdm(zip(per_month["month"], per_month["rows"]), total=len(per_month), desc="Merging months", unit="month")])

show_table(manifest.assign(first_ts=manifest["first_ts"].dt.strftime("%Y-%m-%d %H:%M"), last_ts=manifest["last_ts"].dt.strftime("%Y-%m-%d %H:%M")), 1)

Pemeriksaan: jumlah baris tiap bulan sama dengan server, dan rentang waktu antarbulan tidak tumpang tindih, sehingga penyambungan berurutan tetap terurut menurut waktu.

In [ ]:
assert (manifest["rows"] == manifest["server_rows"]).all(), "A month has a row count that differs from the server."
assert (manifest["first_ts"].iloc[1:].values > manifest["last_ts"].iloc[:-1].values).all(), "Monthly time ranges overlap."
print(f"{len(manifest)} monthly files, {fmt(manifest['rows'].sum())} rows: match the server and do not overlap.")

##### 7.3 Bulanan → satu file final

File bulanan disambung berurutan menjadi satu file. `hive_partitioning=false` mencegah DuckDB menambahkan kolom `year` dan `month` dari nama folder.

In [ ]:
FINAL_PATH = FINAL_DIR / f"mta_subway_hourly_ridership_{min(plan)[:7]}_{max(plan)[:7]}.parquet"
file_list = ", ".join(f"'{month_file(m)}'" for m in per_month["month"])

con.execute(f"COPY (SELECT * FROM read_parquet([{file_list}], hive_partitioning=false)) TO '{FINAL_PATH}' ({COPY_OPTS})")
print(f"Final file : {FINAL_PATH.name}")
print(f"Size       : {fmt(FINAL_PATH.stat().st_size / 1e6, 1)} MB")

#### 8. Pemeriksaan Struktur Awal Dataset

Dataset final dibuka sebagai *view* DuckDB (`mta`) sehingga dapat dikueri dengan SQL tanpa memuat seluruh baris ke memori.

##### 8.1 Dimensi dan tipe data

In [ ]:
con.execute(f"CREATE OR REPLACE VIEW mta AS SELECT * FROM read_parquet('{FINAL_PATH}')")

n_rows = con.sql("SELECT count(*) FROM mta").fetchone()[0]
print(f"Shape: {fmt(n_rows)} rows x {len(COLUMNS)} columns")
show_table(con.sql("DESCRIBE mta").df()[["column_name", "column_type"]].set_axis(["column", "type"], axis=1))

##### 8.2 Contoh baris dan `info`

Lima baris pertama:

In [ ]:
con.sql("SELECT * FROM mta LIMIT 5").df()

`info()` pada sampel acak 100.000 baris (bukan seluruh data):

In [ ]:
con.sql("SELECT * FROM mta USING SAMPLE 100000 ROWS (reservoir, 42)").df().info()

##### 8.3 Nilai kosong

Jumlah `NULL` pada semua kolom dan string kosong pada kolom teks.

In [ ]:
TEXT_COLS = [f.name for f in SCHEMA if pa.types.is_string(f.type)]
q_null = ", ".join(f"count(*) - count({c}) AS {c}" for c in COLUMNS)
q_empty = ", ".join(f"count(*) FILTER (WHERE trim({c}) = '') AS {c}" for c in TEXT_COLS)

nulls = con.sql(f"SELECT {q_null} FROM mta").df().iloc[0]
empty = con.sql(f"SELECT {q_empty} FROM mta").df().iloc[0]
show_table(pd.DataFrame({"column": COLUMNS, "null_count": nulls.values, "empty_string_count": [empty.get(c, 0) for c in COLUMNS]}))

##### 8.4 Kolom kategorik

Jumlah nilai unik tiap kolom teks:

In [ ]:
unique = con.sql("SELECT " + ", ".join(f"count(DISTINCT {c}) AS {c}" for c in TEXT_COLS) + " FROM mta").df().T
show_table(unique.reset_index().set_axis(["column", "unique_values"], axis=1))

Sebaran baris dan total penumpang untuk kolom kategori utama:

In [ ]:
for col in ["transit_mode", "borough", "payment_method", "fare_class_category"]:
    dist = con.sql(f"""
        SELECT {col} AS value, count(*) AS row_count, round(100.0 * count(*) / {n_rows}, 2) AS pct, sum(ridership) AS ridership
        FROM mta GROUP BY 1 ORDER BY 2 DESC""").df()
    print(f"\nDistribution of {col}")
    display(show_table(dist, 2))

##### 8.5 Kolom numerik

Ringkasan statistik `ridership` dan `transfers`.

In [ ]:
def summarize(col):
    return (f"SELECT '{col}' AS variable, min({col}) AS min, quantile_cont({col}, 0.5) AS median, avg({col}) AS mean, "
            f"quantile_cont({col}, 0.95) AS p95, max({col}) AS max, stddev({col}) AS std, sum({col}) AS total FROM mta")


show_table(con.sql(" UNION ALL ".join(map(summarize, ["ridership", "transfers"]))).df(), 2)

##### 8.6 Cakupan waktu

Rentang waktu serta jumlah jam dan hari unik:

In [ ]:
time_span = con.sql("""
    SELECT strftime(min(transit_timestamp), '%Y-%m-%d %H:%M') AS first_ts, strftime(max(transit_timestamp), '%Y-%m-%d %H:%M') AS last_ts,
           count(DISTINCT transit_timestamp) AS unique_hours, count(DISTINCT CAST(transit_timestamp AS DATE)) AS unique_days
    FROM mta""").df()
show_table(time_span)

Jam yang tidak muncul pada rentang tersebut:

In [ ]:
missing_hours = con.sql("""
    SELECT h AS missing_hour
    FROM generate_series((SELECT min(transit_timestamp) FROM mta), (SELECT max(transit_timestamp) FROM mta), INTERVAL 1 HOUR) AS g(h)
    WHERE h NOT IN (SELECT DISTINCT transit_timestamp FROM mta)""").df()
print(f"Missing hours: {len(missing_hours)}")
missing_hours

##### 8.7 Catatan awal untuk tahap berikutnya

Pemeriksaan cepat yang hanya **mencatat** temuan; tidak ada perbaikan pada tahap ini.

In [ ]:
show_table(con.sql("""
    SELECT count(*) FILTER (WHERE transfers < 0) AS negative_transfers,
           count(*) FILTER (WHERE transfers > ridership) AS transfers_exceeding_ridership,
           count(*) FILTER (WHERE ridership <= 0) AS nonpositive_ridership
    FROM mta""").df().T.reset_index().set_axis(["check", "row_count"], axis=1))

Kompleks stasiun yang tercatat dengan lebih dari satu moda:

In [ ]:
con.sql("""
    SELECT station_complex_id, station_complex, string_agg(DISTINCT transit_mode, ', ') AS modes
    FROM mta GROUP BY 1, 2 HAVING count(DISTINCT transit_mode) > 1""").df()

#### 9. Validasi Hasil *Gathering*

Bagian ini membuktikan bahwa dataset final lengkap dan sama dengan sumbernya.

##### 9.1 Perbandingan per hari dengan server

Server menghitung agregat harian dari data aslinya (jumlah baris, total `ridership`, total `transfers`). Hasilnya digabung (`pd.merge`) dengan agregat yang dihitung dari dataset final, berdasarkan tanggal.

In [ ]:
server = pd.DataFrame(aggregate(
    "date_trunc_ymd(transit_timestamp) AS ymd, count(*) AS n, sum(ridership) AS r, sum(transfers) AS t",
    where=f"transit_timestamp <= '{CUTOFF_TS}'", group="ymd"))
server["ymd"] = server["ymd"].str[:10]
server = server[server["ymd"].isin(plan)].astype({"n": int, "r": float, "t": float})

Agregat yang sama dihitung dari dataset final:

In [ ]:
local = con.sql("""
    SELECT strftime(CAST(transit_timestamp AS DATE), '%Y-%m-%d') AS ymd, count(*) AS n, sum(ridership) AS r, sum(transfers) AS t
    FROM mta GROUP BY 1""").df()

Kedua agregat dibandingkan per hari dengan `pd.merge`:

In [ ]:
compare = server.merge(local, on="ymd", how="outer", suffixes=("_server", "_final"), indicator=True)
mismatch = compare[["n_server", "r_server", "t_server"]].values != compare[["n_final", "r_final", "t_final"]].values
n_diff = int(((compare["_merge"] != "both") | mismatch.any(axis=1)).sum())

print(f"Days compared: {len(compare)} | days that differ: {n_diff}")
show_table(compare.head(5).drop(columns="_merge"))

##### 9.2 Keunikan dan urutan baris

Data sudah terurut menurut `SORT_KEYS`, sehingga duplikat pasti berdampingan. Membandingkan tiap baris dengan baris sebelumnya (`lag`) cukup untuk memeriksa duplikat dan urutan, dan lebih hemat memori daripada mengelompokkan 45 juta kunci. Pemeriksaan dilakukan pada tiap file bulanan (kunci lengkap) dan pada file final (`transit_timestamp` tidak boleh mundur).

In [ ]:
def check_order(path):
    keys = ", ".join(SORT_KEYS)
    dupes, unsorted = con.execute(f"""
        WITH t AS (
            SELECT ({keys}) AS k, lag(({keys})) OVER (ORDER BY file_row_number) AS prev
            FROM read_parquet('{path}', file_row_number = true))
        SELECT count(*) FILTER (WHERE k = prev), count(*) FILTER (WHERE k < prev) FROM t""").fetchone()
    return {"dupes": dupes, "unsorted": unsorted}

Pemeriksaan pada tiap file bulanan:

In [ ]:
order_check = pd.DataFrame([{"month": m, **check_order(month_file(m))} for m in tqdm(per_month["month"], desc="Checking order and duplicates", unit="month")])
print(f"Duplicate keys (all monthly files): {order_check['dupes'].sum()}")
print(f"Unsorted rows (monthly files)     : {order_check['unsorted'].sum()}")

Pemeriksaan pada file final:

In [ ]:
final_backward = con.execute(f"""
    WITH t AS (
        SELECT transit_timestamp AS ts, lag(transit_timestamp) OVER (ORDER BY file_row_number) AS prev
        FROM read_parquet('{FINAL_PATH}', file_row_number = true))
    SELECT count(*) FILTER (WHERE ts < prev) FROM t""").fetchone()[0]
print(f"Timestamps going backward (final) : {final_backward}")

##### 9.3 Ringkasan validasi

Seluruh pemeriksaan dikumpulkan dalam satu kamus. Nilai acuan (jumlah baris dan total penumpang) hanya berlaku untuk *snapshot* penuh.

In [ ]:
r_total, t_total = con.sql("SELECT sum(ridership), sum(transfers) FROM mta").fetchone()

checks = {
    "Daily rows = server rows (snapshot)": n_daily == PLAN_ROWS,
    "Monthly rows = server rows": manifest["rows"].sum() == PLAN_ROWS,
    "Final dataset rows = server rows": n_rows == PLAN_ROWS,
    "Per-day aggregates match server (rows, ridership, transfers)": n_diff == 0,
    "No duplicate keys": order_check["dupes"].sum() == 0,
    "Data sorted by key (monthly and final)": order_check["unsorted"].sum() == 0 and final_backward == 0,
    "No null values": nulls.sum() == 0,
}
if TEST_MONTHS is None:
    checks |= {
        "Final rows = reference snapshot value": n_rows == EXPECTED["rows"],
        "Total ridership = reference value": round(r_total) == EXPECTED["ridership"],
        "Total transfers = reference value": round(t_total) == EXPECTED["transfers"],
    }

Hasil pemeriksaan; sel berhenti dengan galat bila ada yang gagal:

In [ ]:
display(show_table(pd.DataFrame({"Check": checks.keys(), "Status": ["PASS" if ok else "FAIL" for ok in checks.values()]})))
assert all(checks.values()), "At least one validation check failed."

#### 10. Dataset Akhir Hasil *Gathering*

Ringkasan dataset akhir yang diserahkan ke tahap *assessing* dan *cleaning*.

In [ ]:
show_table(pd.DataFrame([
    ("File name", FINAL_PATH.name),
    ("Folder", str(FINAL_DIR)),
    ("Format", "Apache Parquet (zstd compression)"),
    ("File size", f"{fmt(FINAL_PATH.stat().st_size / 1e6, 1)} MB"),
    ("Rows", fmt(n_rows)),
    ("Columns", len(COLUMNS)),
    ("Time range", f"{time_span.loc[0, 'first_ts']} to {time_span.loc[0, 'last_ts']}"),
    ("Station complexes", fmt(con.sql("SELECT count(DISTINCT station_complex_id) FROM mta").fetchone()[0])),
    ("Total ridership", fmt(round(r_total))),
    ("Snapshot cutoff", CUTOFF_TS.replace("T", " ")),
], columns=["Attribute", "Value"]))

Folder `/content` dihapus saat sesi berakhir. Centang `SAVE_TO_DRIVE` untuk menyalin dataset final ke Google Drive (Colab akan meminta izin akses).

In [ ]:
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

if SAVE_TO_DRIVE:
    import shutil
    from google.colab import drive

    drive.mount("/content/drive")
    shutil.copy2(FINAL_PATH, "/content/drive/MyDrive/")

##### Ringkasan

- Data dikumpulkan dari satu sumber (SODA API `5wq4-mkjj`) dengan *snapshot* tetap sampai 2026-09-16 23:00.
- 624 potongan harian digabung menjadi 21 file bulanan lalu satu file final: **45.243.727 baris × 12 kolom**.
- Validasi terhadap server lulus: jumlah baris dan total nilai per hari sama, tanpa duplikat kunci dan tanpa nilai *null*.
- Catatan awal: 226 baris `transfers` negatif, satu jam tidak muncul (2026-03-08 02:00), dan satu stasiun (St George, ID 501) tercatat dengan dua moda.

**Selanjutnya:** *assessing* dan *cleaning* pada dataset final ini, lalu pengumpulan sumber *streaming* (GTFS-RT).